In [2]:
import pandas as pd
from pathlib import Path

In [ ]:
RAW_FILE = "../data/raw/variant_summary.csv"
SAVE_FILE = "../data/processed/IBDC_filtered.csv"

In [4]:
import os

size = os.path.getsize(RAW_FILE) / (1024**3)
print(f"File size: {size:.2f} GB")

File size: 3.68 GB


In [5]:
chunksize = 500000

filtered_chunks = []

for chunk in pd.read_csv(
        RAW_FILE,
        chunksize=chunksize,
        dtype=str,
        low_memory=False):

    # keep only human genome build
    chunk = chunk[chunk["Assembly"] == "GRCh38"]

    # remove rows with no gene symbol
    chunk = chunk[chunk["GeneSymbol"].notna()]

    # keep only meaningful clinical labels
    valid_labels = [
        "Pathogenic",
        "Likely pathogenic",
        "Benign",
        "Likely benign"
    ]

    chunk = chunk[chunk["ClinicalSignificance"].isin(valid_labels)]

    # select important columns
    chunk = chunk[[
        "VariationID",
        "GeneSymbol",
        "Chromosome",
        "Start",
        "ReferenceAllele",
        "AlternateAllele",
        "ClinicalSignificance",
        "Type"
    ]]

    filtered_chunks.append(chunk)

In [6]:
clinvar = pd.concat(filtered_chunks, ignore_index=True)

print("Total filtered variants:", len(clinvar))
clinvar.head()

Total filtered variants: 1591701


,VariationID,GeneSymbol,Chromosome,Start,ReferenceAllele,AlternateAllele,ClinicalSignificance,Type
0,3,AP5Z1,7,4787730,na,na,Pathogenic,Deletion
1,5,FOXRED1,11,126275389,na,na,Pathogenic,single nucleotide variant
2,6,FOXRED1,11,126277517,na,na,Likely pathogenic,single nucleotide variant
3,14,HFE,6,26093008,na,na,Benign,single nucleotide variant
4,18,HFE,6,26093215,na,na,Pathogenic,single nucleotide variant


In [7]:
clinvar.to_csv(SAVE_FILE, index=False)

print("Saved:", SAVE_FILE)

Saved: ../data/processed/clinvar_filtered.csv


In [8]:
clinvar["ClinicalSignificance"].value_counts()

ClinicalSignificance
Likely benign        1076934
Benign                211577
Pathogenic            191935
Likely pathogenic     111255
Name: count, dtype: int64

In [9]:
gene_features = clinvar.groupby("GeneSymbol").agg(
    total_variants=("VariationID", "count"),
    pathogenic_variants=("ClinicalSignificance",
                         lambda x: x.str.contains("Pathogenic").sum()),
    benign_variants=("ClinicalSignificance",
                     lambda x: x.str.contains("Benign").sum())
).reset_index()

In [10]:
gene_features.to_csv(
    "../data/processed/gene_mutation_features.csv",
    index=False
)

gene_features.head()

,GeneSymbol,total_variants,pathogenic_variants,benign_variants
0,-,737,34,420
1,A-GAMMA3'E;BGLT3;HBE1;HBG1;HBG2;HS-E1;LOC10609...,1,1,0
2,A1BG,2,0,0
3,A1CF,6,0,4
4,A2M,40,0,18
